In [0]:
gold_tbl_customers= 'superstore_catalog.superstore_gold.dim_customers'

In [0]:
# One-current-row rule (MOST IMPORTANT)
# Each customer must have exactly ONE current row
spark.sql(f"""
SELECT customer_id, COUNT(*) cnt
FROM superstore_catalog.superstore_gold.dim_customers
WHERE is_current = true
GROUP BY customer_id
HAVING cnt != 1
""").show()

In [0]:
# SCD2 timeline correctness- No overlapping date ranges
spark.sql(f"""
SELECT customer_id
FROM superstore_catalog.superstore_gold.dim_customers
GROUP BY customer_id
HAVING COUNT(*) > 1
AND MAX(effective_to) IS NOT NULL
AND MIN(effective_from) > MAX(effective_to)
""").show()


In [0]:
# Late-arriving data validation- Pick one customer_id:
# Check:
# Older change appears before
# Newer record is is_current = true
# Previous record has effective_to

spark.sql(f"""
SELECT *
FROM superstore_catalog.superstore_gold.dim_customers
WHERE customer_id = 'PR-18880'
ORDER BY effective_from
""").display()


In [0]:
# Idempotency test (CRITICAL)
# Run the full notebook TWICE
# Then check:Should return ZERO rows

spark.sql(f"""
SELECT customer_id, effective_from, COUNT(*)
FROM superstore_catalog.superstore_gold.dim_customers
GROUP BY customer_id, effective_from
HAVING COUNT(*) > 1
""").display()


In [0]:
# Soft delete validation-Delete a customer from Silver:
spark.sql("""
DELETE FROM superstore_catalog.superstore_silver.customers
WHERE customer_id = 'PR-18880'
""")


Run pipeline again, then:is_current = false
effective_to populated

In [0]:
spark.sql(f"""
SELECT *
FROM superstore_catalog.superstore_gold.dim_customers
WHERE customer_id = 'PR-18880'
""").display()


In [0]:
# Hash change validation-Update a Silver attribute:

In [0]:
spark.sql("""
UPDATE superstore_catalog.superstore_silver.customers
SET city = 'NEW CITY'
WHERE customer_id = 'PR-18880'
""")


Run pipeline again:

In [0]:
spark.sql(f"""
SELECT customer_id, city, is_current
FROM superstore_catalog.superstore_gold.dim_customers
WHERE customer_id = 'PR-18880'
ORDER BY effective_from
""").display()

# Old record expired
# New record current

In [0]:
spark.table('superstore_catalog.superstore_gold.dim_customers').count()

In [0]:
spark.table('superstore_catalog.superstore_silver.customers').count()

In [0]:
# Delete rows where customer_id=90 AND city='oo'
spark.sql("""
DELETE FROM superstore_catalog.superstore_gold.dim_customers
WHERE customer_id = 'PR-18880' AND city = 'NEW CITY'
""")



In [0]:
spark.sql("""
update superstore_catalog.superstore_silver.customers
set city='Oakland'
where customer_id= 'PR-18880'
""")



In [0]:
spark.sql("""
select city 
from superstore_catalog.superstore_silver.customers
where customer_id= 'PR-18880'
""").display()

In [0]:
spark.sql(f"""
SELECT customer_id, city, is_current
FROM superstore_catalog.superstore_gold.dim_customers
WHERE customer_id = 'PR-18880'
ORDER BY effective_from
""").display()

# Old record expired
# New record current

In [0]:
spark.table('superstore_catalog.superstore_gold.dim_customers').count()

In [0]:
spark.table('superstore_catalog.superstore_silver.customers').count()

In [0]:
gold_df = spark.table("superstore_catalog.superstore_gold.dim_customers")
gold_df.count()  # total historical + current rows
gold_df.filter(col("is_current") == True).count()  # current only


In [0]:
from pyspark.sql.functions import col
spark.table('superstore_catalog.superstore_gold.dim_customers')\
    .filter(col('customer_id')=='PR-18880').display()

In [0]:
from pyspark.sql.functions import col

silver_df = spark.table("superstore_catalog.superstore_silver.customers")
gold_df = spark.table("superstore_catalog.superstore_gold.dim_customers")

# Rows in Silver not in Gold (by customer_id) at all
missing_in_gold = silver_df.join(gold_df, on="customer_id", how="left_anti")
missing_in_gold.display()

# Rows in Silver where Gold has is_current = false
expired_in_gold = silver_df.join(
    gold_df.filter(col("is_current") == False),
    on="customer_id",
    how="left_semi"
)
expired_in_gold.display()


In [0]:
spark.sql(f"""
SELECT *
FROM superstore_catalog.superstore_gold.dim_customers
WHERE customer_id = 'PR-18880'
""").display()

In [0]:
spark.sql(f"""
SELECT *
FROM superstore_catalog.superstore_silver.customers
WHERE customer_id = 'PR-18880'
""").display()

#products-test

In [0]:
# One-current-row rule (MOST IMPORTANT)
# Each customer must have exactly ONE current row
spark.sql(f"""
SELECT product_id, COUNT(*) cnt
FROM superstore_catalog.superstore_gold.dim_products
WHERE is_current = true
GROUP BY product_id
HAVING cnt != 1
""").show()

In [0]:
silver_tbl = "superstore_catalog.superstore_silver.products"
spark.table(silver_tbl).count()

In [0]:
spark.table('superstore_catalog.superstore_gold.dim_products').count()

In [0]:
spark.table('superstore_catalog.superstore_gold.dim_products').count()

In [0]:
spark.table('superstore_catalog.superstore_gold.dim_products').count()

In [0]:
silver_count = spark.table("superstore_catalog.superstore_silver.products").count()
gold_count = spark.table("superstore_catalog.superstore_gold.dim_products").count()

print(f"Silver: {silver_count}, Gold: {gold_count}")


In [0]:
silver_ids = spark.table("superstore_catalog.superstore_silver.products").select("product_id")
gold_ids = spark.table("superstore_catalog.superstore_gold.dim_products").select("product_id")
missing_in_gold = silver_ids.join(gold_ids, on="product_id", how="left_anti")
missing_in_gold.show()


facts_orders

In [0]:
#count silver and gold rows
silver_count = spark.table("superstore_catalog.superstore_silver.orders").count()
gold_count = spark.table("superstore_catalog.superstore_gold.facts_orders").count()

print(f"Silver: {silver_count}, Gold: {gold_count}")


In [0]:
# Check recent load timestamps
from pyspark.sql.functions import col
spark.table("superstore_catalog.superstore_gold.facts_orders").orderBy(col("load_ts").desc()).limit(10).display()



In [0]:
# Compare hashes
from pyspark.sql.functions import col
gold_df = spark.table("superstore_catalog.superstore_gold.facts_orders").alias("gold")
silver_df = spark.table("superstore_catalog.superstore_silver.orders").alias("silver")

gold_df.join(silver_df, on="order_id") \
    .filter(col("gold.silver_order_hash_id") != col("silver.silver_order_hash_id")) \
    .display()



sales


In [0]:
silver_tbl = "superstore_catalog.superstore_silver.sales"
spark.table(silver_tbl).count()

In [0]:
silver_tbl = "superstore_catalog.superstore_gold.facts_sales"
spark.table(silver_tbl).count()